In [1]:
# ==========================================
# 1. IMPORTS & DEPENDENCIES
# ==========================================
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    precision_score,
    r2_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier, XGBRegressor

# ==========================================
# 2. DYNAMIC & PORTABLE FILE LOADING
# ==========================================
file_name = 'Cleaned_FAANG_Data 1.xlsx'

if os.path.exists(file_name):
    df = pd.read_excel(file_name)
    print(f"Loaded '{file_name}' from local directory.")
else:
    try:
        from google.colab import files
        print(f"'{file_name}' not found locally. Please upload the file:")
        uploaded = files.upload()
        file_name = list(uploaded.keys())[0]
        df = pd.read_excel(file_name)
    except ImportError:
        raise FileNotFoundError(
            f"Could not find '{file_name}' in the current directory. "
            "Please ensure the Excel file is placed in the same folder as this notebook."
        )

# ==========================================
# 3. EXPLORATORY DATA ANALYSIS (EDA)
# ==========================================
print("\n--- DATA OVERVIEW ---")
print("Dataset Shape:", df.shape)
df.info()

print("\n--- MISSING VALUES & DUPLICATES ---")
print("Missing values per column:\n", df.isnull().sum())
print("Duplicate rows count:", df.duplicated().sum())

# Sort data sequentially by Source/Company and Date
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values(by=['Source.Name', 'Date']).reset_index(drop=True)

# Company Stock Price & Return Summary Statistics
company_stats = df.groupby('Source.Name')['Adj Close'].agg(['mean', 'median', 'std', 'min', 'max'])
print("\n--- COMPANY STATS (Adj Close) ---")
print(company_stats.round(3))

df['EDA_Daily_Return'] = df.groupby('Source.Name')['Adj Close'].pct_change()
return_stats = df.groupby('Source.Name')['EDA_Daily_Return'].agg(['mean', 'std', 'min', 'max'])
print("\n--- DAILY RETURN STATS ---")
print(return_stats.round(3))

performance = df.groupby('Source.Name').agg(
    First_Price=('Adj Close', 'first'),
    Last_Price=('Adj Close', 'last')
)
performance['Total Return (%)'] = ((performance['Last_Price'] - performance['First_Price']) / performance['First_Price']) * 100
print("\n--- LONG-TERM PERFORMANCE ---")
print(performance.round(2))

# Calculate Moving Averages for Visual Inspection
df['SMA_50_Adj'] = df.groupby('Source.Name')['Adj Close'].transform(lambda x: x.rolling(50).mean())
df['SMA_200_Adj'] = df.groupby('Source.Name')['Adj Close'].transform(lambda x: x.rolling(200).mean())

# ==========================================
# 4. FEATURE ENGINEERING FOR MODELING
# ==========================================
df['Close_Lag1'] = df.groupby('Source.Name')['Close'].shift(1)
df['Close_Lag5'] = df.groupby('Source.Name')['Close'].shift(5)
df['SMA_10'] = df.groupby('Source.Name')['Close'].transform(lambda x: x.rolling(window=10).mean())
df['SMA_50'] = df.groupby('Source.Name')['Close'].transform(lambda x: x.rolling(window=50).mean())
df['High_Low_Range'] = df['High'] - df['Low']
df['Daily_Return'] = df.groupby('Source.Name')['Close'].pct_change() * 100

# Targets: Regress next day price & Classify direction
df['Target_Price'] = df.groupby('Source.Name')['Close'].shift(-1)
df['Target_Direction'] = (df['Target_Price'] > df['Close']).astype(int)

# Drop missing values generated by lag & rolling operations
df = df.dropna().reset_index(drop=True)

# Class Imbalance Check
print("\n--- CLASS IMBALANCE CHECK ---")
class_counts = df['Target_Direction'].value_counts(normalize=True)
print(class_counts)

num_neg = (df['Target_Direction'] == 0).sum()
num_pos = (df['Target_Direction'] == 1).sum()
scale_pos_weight_val = num_neg / num_pos if num_pos > 0 else 1.0

# ==========================================
# 5. TRAIN / TEST SPLIT
# ==========================================
features = ['Close_Lag1', 'Close_Lag5', 'SMA_10', 'SMA_50', 'High_Low_Range', 'Daily_Return']
X = df[features]
y_reg = df['Target_Price']
y_cls = df['Target_Direction']

train_size = int(len(df) * 0.8)

X_train, X_test = X.iloc[:train_size], X.iloc[train_size:]
y_train_reg, y_test_reg = y_reg.iloc[:train_size], y_reg.iloc[train_size:]
y_train_cls, y_test_cls = y_cls.iloc[:train_size], y_cls.iloc[train_size:]

# ==========================================
# 6. REGRESSION MODEL TRAINING
# ==========================================
reg_models = {
    'Linear Regression': LinearRegression(),
    'Random Forest Regressor': RandomForestRegressor(n_estimators=100, random_state=42),
    'XGBoost Regressor': XGBRegressor(n_estimators=100, learning_rate=0.05, max_depth=5, random_state=42)
}

reg_results, trained_reg_pipelines = [], {}
for name, model in reg_models.items():
    pipeline = Pipeline([('scaler', StandardScaler()), ('model', model)])
    pipeline.fit(X_train, y_train_reg)
    preds = pipeline.predict(X_test)

    reg_results.append({
        'Model': name,
        'MAE ($)': round(mean_absolute_error(y_test_reg, preds), 2),
        'RMSE ($)': round(np.sqrt(mean_squared_error(y_test_reg, preds)), 2),
        'R2 Score': round(r2_score(y_test_reg, preds), 4)
    })
    trained_reg_pipelines[name] = pipeline

reg_df = pd.DataFrame(reg_results).sort_values(by='MAE ($)').reset_index(drop=True)
print('\n--- REGRESSION RESULTS ---')
print(reg_df)

# ==========================================
# 7. CLASSIFICATION MODEL TRAINING
# ==========================================
cls_models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced'),
    'Random Forest Classifier': RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=42),
    'XGBoost Classifier': XGBClassifier(
        n_estimators=100,
        learning_rate=0.05,
        max_depth=5,
        scale_pos_weight=scale_pos_weight_val,
        random_state=42
    )
}

cls_results, trained_cls_pipelines = [], {}
for name, model in cls_models.items():
    pipeline = Pipeline([('scaler', StandardScaler()), ('model', model)])
    pipeline.fit(X_train, y_train_cls)
    preds = pipeline.predict(X_test)

    cls_results.append({
        'Model': name,
        'Accuracy': round(accuracy_score(y_test_cls, preds), 4),
        'Precision': round(precision_score(y_test_cls, preds), 4),
        'Recall': round(recall_score(y_test_cls, preds), 4),
        'F1 Score': round(f1_score(y_test_cls, preds), 4)
    })
    trained_cls_pipelines[name] = pipeline

cls_df = pd.DataFrame(cls_results).sort_values(by='F1 Score', ascending=False).reset_index(drop=True)
print('\n--- CLASSIFICATION RESULTS ---')
print(cls_df)

# ==========================================
# 8. HYPERPARAMETER TUNING (XGBOOST)
# ==========================================
print('\n--- RUNNING HYPERPARAMETER TUNING (XGBOOST) ---')
tscv = TimeSeriesSplit(n_splits=3)

xgb_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', XGBClassifier(random_state=42, scale_pos_weight=scale_pos_weight_val))
])

param_grid = {
    'model__n_estimators': [50, 100],
    'model__max_depth': [3, 5],
    'model__learning_rate': [0.01, 0.05]
}

grid_search = GridSearchCV(
    estimator=xgb_pipe,
    param_grid=param_grid,
    cv=tscv,
    scoring='f1',
    n_jobs=-1
)
grid_search.fit(X_train, y_train_cls)

print('Best Parameters for XGBoost Classifier:', grid_search.best_params_)
tuned_xgb_cls = grid_search.best_estimator_

# ==========================================
# 9. EXPORT PREDICTIONS AND MODELS
# ==========================================
best_reg_pipeline = trained_reg_pipelines[reg_df.iloc[0]['Model']]
best_cls_pipeline = tuned_xgb_cls

df_test = df.iloc[train_size:].copy()
df_test['Predicted_Close_Price'] = best_reg_pipeline.predict(X_test)
df_test['Predicted_Market_Direction'] = best_cls_pipeline.predict(X_test)

df_test.to_csv('FAANG_Predictions_For_BI.csv', index=False)
joblib.dump(best_reg_pipeline, 'faang_best_regression_model.pkl')
joblib.dump(best_cls_pipeline, 'faang_best_classification_model.pkl')

print('\nExecution Completed & Files Saved Successfully!')

Loaded 'Cleaned_FAANG_Data 1.xlsx' from local directory.

--- DATA OVERVIEW ---
Dataset Shape: (26565, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 26565 entries, 0 to 26564
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   Source.Name  26565 non-null  object        
 1   Date         26565 non-null  datetime64[ns]
 2   Open         26565 non-null  float64       
 3   High         26565 non-null  float64       
 4   Low          26565 non-null  float64       
 5   Close        26565 non-null  float64       
 6   Adj Close    26565 non-null  float64       
 7   Volume       26565 non-null  int64         
 8   Year         26565 non-null  int64         
 9   Month        26565 non-null  int64         
 10  Change(%)    26565 non-null  float64       
 11  Range        26565 non-null  float64       
dtypes: datetime64[ns](1), float64(7), int64(3), object(1)
memory usage: 2.4+ MB

--- MISSING VALU